![image.png](attachment:44d2f751-c0ff-48be-8e86-77b4f69628a6.png)

#### Passo 1: Instalar bibliotecas

In [2]:
#!pip install requests
#!pip install beautifulsoup4
#!pip install sqlite3
#!pip install pandas

#### Passo 2: Importar bibliotecas

In [3]:
import requests
from bs4 import BeautifulSoup
import sqlite3
import pandas as pd

#### Passo 3: Fromatar link de busca

Neste exemplo utilizaremos um site público de testes de scraping

http://books.toscrape.com (feito exatamente para treinar scraping).

In [4]:
# URL base
url = "https://books.toscrape.com/catalogue/category/books_1/page-1.html"

#### Passo 4: Determinar variáveis que para armazenamento
Neste exemplo, iremos utilizar:
- Título
- Preços
- Estoque

In [5]:
titulos, precos, estoque = [], [], []

#### Passo 5: Obter informações do link

In [6]:
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

#### Passo 5.1: Obter informações dos produtos

In [7]:
# Cada produto está dentro de article.product_pod
livros = soup.find_all("article", class_="product_pod")

In [8]:
livros[0].h3.a['title']

'A Light in the Attic'

In [9]:
# Preço
livros[0].find("p", class_="price_color").text.replace("Â£", "")

'51.77'

In [10]:
# Estoque
livros[0].find("p", class_="instock availability").text.strip()
# A função strip()em Python serve para remover espaços em branco ou caracteres específicos do início e/ou fim de uma string

'In stock'

#### Passo 6: Coletar dados em escala
Neste exemplo, vamos coletar dados das 5 primeiras páginas

In [11]:
for page in range(1, 5):
    response = requests.get(f"https://books.toscrape.com/catalogue/category/books_1/page-{page}.html")
    soup = BeautifulSoup(response.text, "html.parser")

    # Cada produto está dentro de article.product_pod
    livros = soup.find_all("article", class_="product_pod")

    for livro in livros:
        # Título
        titulo = livro.h3.a["title"]
        titulos.append(titulo)

        # Preço
        preco = livro.find("p", class_="price_color").text.replace("Â£", "")
        precos.append(preco)

        # Estoque
        estoque_info = livro.find("p", class_="instock availability").text.strip()
        estoque.append(estoque_info)

print("Coleta concluída!")

Coleta concluída!


#### Passo 7: Criar DataFrame

In [12]:
df = pd.DataFrame({"titulo": titulos, "preco": precos, "estoque": estoque})
df.head()

,titulo,preco,estoque
0,A Light in the Attic,51.77,In stock
1,Tipping the Velvet,53.74,In stock
2,Soumission,50.10,In stock
3,Sharp Objects,47.82,In stock
4,Sapiens: A Brief History of Humankind,54.23,In stock


#### Passo 8: Criar Banco de Dados

In [13]:
conn = sqlite3.connect("library.db")

#### Passo 9: DDL – Data Definition Language
DDL serve para **definir a estrutura do banco de dados**: criar, alterar e excluir tabelas.

- `CREATE` → cria tabelas e estruturas  
- `ALTER` → modifica a estrutura de tabelas  
- `DROP` → apaga tabelas ou bancos de dados  


In [14]:
cursor = conn.cursor()

In [15]:
create = """
          CREATE TABLE IF NOT EXISTS books(
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT NOT NULL,
            price REAL NOT NULL,
            stock TEXT NOT NULL
          );
         """
cursor.execute(create)

##### 9.1: Listar todas tabelas do banco
Neste exemplo iremos utilizaremos a sintaxe `SELECT name FROM sqlite_master;`

- `sqlite_master` é uma tabela especial **interna** do SQLite que guarda **metadados** sobre o banco de dados: tabelas, índices, views e triggers
- `SELECT name` seleciona apenas o nome (`name`) dos objetos armazenados no `sqlite_master`

In [16]:
cursor.execute("SELECT name FROM sqlite_master")
print(cursor.fetchall())

[('books',), ('sqlite_sequence',)]


Como pegar os resultados usando o método `fetch` (buscar)
- `fetchone()` Retorna apenas uma linha do resultado (a próxima disponível). Se não houver mais nada, retorna `None`
- `fetchmany(n)` Retorna as próximas n linhas do resultado.
- `fetchall()` Retorna todas as linhas restantes do resultado em uma lista de tuplas.

In [17]:
cursor.execute("SELECT name FROM sqlite_master")
print(cursor.fetchone())
print(cursor.fetchone())
print(cursor.fetchone())

('books',)
('sqlite_sequence',)
None


In [18]:
cursor.execute("SELECT name FROM sqlite_master")
print(cursor.fetchmany(2))

[('books',), ('sqlite_sequence',)]


##### 9.2: Verificar o esquema da tabela (estrutura das colunas)

In [19]:
cursor.execute("PRAGMA table_info(books)")
columns = cursor.fetchall()
print(columns)

for i in columns:
  print(i[1], i[2])

[(0, 'id', 'INTEGER', 0, None, 1), (1, 'title', 'TEXT', 1, None, 0), (2, 'price', 'REAL', 1, None, 0), (3, 'stock', 'TEXT', 1, None, 0)]
id INTEGER
title TEXT
price REAL
stock TEXT


In [20]:
cursor.execute("PRAGMA table_info(books)")
for row in cursor.fetchall():
  print(row)

(0, 'id', 'INTEGER', 0, None, 1)
(1, 'title', 'TEXT', 1, None, 0)
(2, 'price', 'REAL', 1, None, 0)
(3, 'stock', 'TEXT', 1, None, 0)


(`cid` , `name` ,`type` , `notnull` , `dflt_value` , `pk`)

- `cid` índice da coluna
- `name` nome da coluna
- `type` tipo de dado (TEXT, REAL, etc.)
- `notnull` se pode ser NULL (0 = sim, 1 = não)
- `dflt_value` valor padrão (se houver)
- `pk` se é chave primária (1 = sim)

##### 9.3: Alterar tabela
Nesta etapa utilizaremos o comando `ALTER TABLE` informar a alteração da tabela e o comando `ADD COLUMN` para especificar a alteração

In [21]:
alter_table = "ALTER TABLE books ADD COLUMN category TEXT;"
cursor.execute(alter_table)

In [22]:
cursor.execute("PRAGMA table_info(books)")
columns = cursor.fetchall()

for i in columns:
  print(i[1], i[2])

id INTEGER
title TEXT
price REAL
stock TEXT
category TEXT


##### 9.4: Verifica o equema da tabela após incluir coluna coluna

##### 9.5: Remover uma coluna
Nesta etapa utilizaremos o comando `ALTER TABLE` informar a alteração da tabela e o comando `DROP COLUMN` para especificar a alteração

In [23]:
alter_table = "ALTER TABLE books DROP COLUMN category;"
cursor.execute(alter_table)



##### 9.6: Verifica o equema da tabela após incluir coluna coluna

In [24]:
cursor.execute("PRAGMA table_info(books)")
for row in cursor.fetchall():
  print(row)

(0, 'id', 'INTEGER', 0, None, 1)
(1, 'title', 'TEXT', 1, None, 0)
(2, 'price', 'REAL', 1, None, 0)
(3, 'stock', 'TEXT', 1, None, 0)


##### 9.7: Excluir tabela
Primeiro vamos criar uma tabela adicional

In [25]:
create = """
          CREATE TABLE IF NOT EXISTS products(
            id INTEGER PRIMARY KEY AUTOINCREMENT
          );
         """
cursor.execute(create)

##### Listar todas tabelas do banco

In [26]:
cursor.execute("SELECT name FROM sqlite_master")
print(cursor.fetchmany(10))

[('books',), ('sqlite_sequence',), ('products',)]


##### Remover tabela do banco

In [27]:
query = "DROP TABLE products;"
cursor.execute(query)
conn.commit()

##### Listar todas tabelas do banco

#### Passo 10: DML – Data Manipulation Language

DML serve para **manipular os dados** dentro das tabelas:

- `INSERT` → inserir registros  
- `UPDATE` → atualizar registros  
- `DELETE` → excluir registros



##### 10.1 `INSERT` (inserir dados)

In [28]:
for _, row in df.iterrows():
  cursor.execute("""
    INSERT INTO books (title, price, stock) VALUES (?, ?, ?)
  """, (row["titulo"], row["preco"], row["estoque"]))

conn.commit()

##### Verifica quantos registros foram inseridos

In [29]:
cursor.execute("SELECT COUNT(*) FROM books")
cursor.fetchone()

(80,)

##### Verifica uma amostra dos registros inseridos

In [30]:
cursor.execute("SELECT * FROM books LIMIT 10")
cursor.fetchall()

[(1, 'A Light in the Attic', 51.77, 'In stock'),
 (2, 'Tipping the Velvet', 53.74, 'In stock'),
 (3, 'Soumission', 50.1, 'In stock'),
 (4, 'Sharp Objects', 47.82, 'In stock'),
 (5, 'Sapiens: A Brief History of Humankind', 54.23, 'In stock'),
 (6, 'The Requiem Red', 22.65, 'In stock'),
 (7, 'The Dirty Little Secrets of Getting Your Dream Job', 33.34, 'In stock'),
 (8,
  'The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull',
  17.93,
  'In stock'),
 (9,
  'The Boys in the Boat: Nine Americans and Their Epic Quest for Gold at the 1936 Berlin Olympics',
  22.6,
  'In stock'),
 (10, 'The Black Maria', 52.15, 'In stock')]

##### 10.2: `UPDATE` (atualizar dados)

In [31]:
update = """
  UPDATE books
  SET title = UPPER(title)
"""
cursor.execute(update)

##### Verifica uma amostra dos registros inseridos

In [32]:
cursor.execute("SELECT * FROM books")
cursor.fetchmany(10)

[(1, 'A LIGHT IN THE ATTIC', 51.77, 'In stock'),
 (2, 'TIPPING THE VELVET', 53.74, 'In stock'),
 (3, 'SOUMISSION', 50.1, 'In stock'),
 (4, 'SHARP OBJECTS', 47.82, 'In stock'),
 (5, 'SAPIENS: A BRIEF HISTORY OF HUMANKIND', 54.23, 'In stock'),
 (6, 'THE REQUIEM RED', 22.65, 'In stock'),
 (7, 'THE DIRTY LITTLE SECRETS OF GETTING YOUR DREAM JOB', 33.34, 'In stock'),
 (8,
  'THE COMING WOMAN: A NOVEL BASED ON THE LIFE OF THE INFAMOUS FEMINIST, VICTORIA WOODHULL',
  17.93,
  'In stock'),
 (9,
  'THE BOYS IN THE BOAT: NINE AMERICANS AND THEIR EPIC QUEST FOR GOLD AT THE 1936 BERLIN OLYMPICS',
  22.6,
  'In stock'),
 (10, 'THE BLACK MARIA', 52.15, 'In stock')]

##### 10.3: Inserir um novo registro no banco

In [33]:
#(0, 'id', 'INTEGER', 0, None, 1)
#(1, 'title', 'TEXT', 1, None, 0)
#(2, 'price', 'REAL', 1, None, 0)
#(3, 'stock', 'TEXT', 1, None, 0)

In [34]:
insert = """
  INSERT INTO books (title, price, stock)
  VALUES (
    'Super Python',
    10.5,
    'In stock'
  );
"""
cursor.execute(insert)

##### Verifica quantos registros existem no banco

In [35]:
cursor.execute("SELECT COUNT(*) FROM books")
cursor.fetchone()

(81,)

##### Verifica uma amostra dos últimos registros inseridos

In [36]:
cursor.execute("SELECT * FROM books ORDER BY 1 DESC LIMIT 1")
cursor.fetchall()

[(81, 'Super Python', 10.5, 'In stock')]

##### 10.4: `DELETE` (exclui registros)

In [37]:
cursor.execute("DELETE FROM books WHERE id = 81")
conn.commit()

##### Verifica uma amostra dos últimos registros inseridos

In [38]:
cursor.execute("SELECT * FROM books ORDER BY 1 DESC LIMIT 10")
cursor.fetchall()

[(80,
  'RAT QUEENS, VOL. 3: DEMONS (RAT QUEENS (COLLECTED EDITIONS) #11-15)',
  50.4,
  'In stock'),
 (79,
  'RESKILLING AMERICA: LEARNING TO LABOR IN THE TWENTY-FIRST CENTURY',
  19.83,
  'In stock'),
 (78, 'SAGA, VOLUME 5 (SAGA (COLLECTED EDITIONS) #5)', 51.04, 'In stock'),
 (77, 'SAGA, VOLUME 6 (SAGA (COLLECTED EDITIONS) #6)', 25.02, 'In stock'),
 (76, 'SECURITY', 39.25, 'In stock'),
 (75, 'SOUL READER', 39.58, 'In stock'),
 (74,
  'SPARK JOY: AN ILLUSTRATED MASTER CLASS ON THE ART OF ORGANIZING AND TIDYING UP',
  41.83,
  'In stock'),
 (73,
  "THE ACTIVIST'S TAO TE CHING: ANCIENT ADVICE FOR A MODERN REVOLUTION",
  32.24,
  'In stock'),
 (72,
  'THE AGE OF GENIUS: THE SEVENTEENTH CENTURY AND THE BIRTH OF THE MODERN MIND',
  19.73,
  'In stock'),
 (71, 'THE ART FORGER', 40.76, 'In stock')]

#### Passo 11: DQL serve para **consultar dados** no banco:

- `SELECT`  
- `WHERE`  
- `ORDER BY`  
- `GROUP BY`  
- Funções de agregação (`COUNT`, `AVG`, `MAX`, `MIN`, `SUM`)

##### Consulta simples com `fetchone`

In [39]:
cursor.execute("SELECT * FROM books")
cursor.fetchone()

(1, 'A LIGHT IN THE ATTIC', 51.77, 'In stock')

##### Consulta simples com `fetchmany()`

In [40]:
cursor.fetchmany(10)

[(2, 'TIPPING THE VELVET', 53.74, 'In stock'),
 (3, 'SOUMISSION', 50.1, 'In stock'),
 (4, 'SHARP OBJECTS', 47.82, 'In stock'),
 (5, 'SAPIENS: A BRIEF HISTORY OF HUMANKIND', 54.23, 'In stock'),
 (6, 'THE REQUIEM RED', 22.65, 'In stock'),
 (7, 'THE DIRTY LITTLE SECRETS OF GETTING YOUR DREAM JOB', 33.34, 'In stock'),
 (8,
  'THE COMING WOMAN: A NOVEL BASED ON THE LIFE OF THE INFAMOUS FEMINIST, VICTORIA WOODHULL',
  17.93,
  'In stock'),
 (9,
  'THE BOYS IN THE BOAT: NINE AMERICANS AND THEIR EPIC QUEST FOR GOLD AT THE 1936 BERLIN OLYMPICS',
  22.6,
  'In stock'),
 (10, 'THE BLACK MARIA', 52.15, 'In stock'),
 (11, 'STARVING HEARTS (TRIANGULAR TRADE TRILOGY, #1)', 13.99, 'In stock')]

##### Consulta simples com `fetchall()`

In [41]:
cursor.fetchall()

[(12, "SHAKESPEARE'S SONNETS", 20.66, 'In stock'),
 (13, 'SET ME FREE', 17.46, 'In stock'),
 (14,
  "SCOTT PILGRIM'S PRECIOUS LITTLE LIFE (SCOTT PILGRIM #1)",
  52.29,
  'In stock'),
 (15, 'RIP IT UP AND START AGAIN', 35.02, 'In stock'),
 (16,
  'OUR BAND COULD BE YOUR LIFE: SCENES FROM THE AMERICAN INDIE UNDERGROUND, 1981-1991',
  57.25,
  'In stock'),
 (17, 'OLIO', 23.88, 'In stock'),
 (18,
  'MESAERION: THE BEST SCIENCE FICTION STORIES 1800-1849',
  37.59,
  'In stock'),
 (19, 'LIBERTARIANISM FOR BEGINNERS', 51.33, 'In stock'),
 (20, "IT'S ONLY THE HIMALAYAS", 45.17, 'In stock'),
 (21, 'IN HER WAKE', 12.84, 'In stock'),
 (22, 'HOW MUSIC WORKS', 37.32, 'In stock'),
 (23,
  'FOOLPROOF PRESERVING: A GUIDE TO SMALL BATCH JAMS, JELLIES, PICKLES, CONDIMENTS, AND MORE: A FOOLPROOF GUIDE TO MAKING SMALL BATCH JAMS, JELLIES, PICKLES, CONDIMENTS, AND MORE',
  30.52,
  'In stock'),
 (24, 'CHASE ME (PARIS NIGHTS #2)', 25.27, 'In stock'),
 (25, 'BLACK DUST', 34.53, 'In stock'),
 (26, 'BIRDSONG: 

#### `WHERE`
##### Consulta com filtro: Livros com preço abaixo de 20

In [42]:
cursor.execute("SELECT * FROM books WHERE price < 20")
cursor.fetchall()

[(8,
  'THE COMING WOMAN: A NOVEL BASED ON THE LIFE OF THE INFAMOUS FEMINIST, VICTORIA WOODHULL',
  17.93,
  'In stock'),
 (11, 'STARVING HEARTS (TRIANGULAR TRADE TRILOGY, #1)', 13.99, 'In stock'),
 (13, 'SET ME FREE', 17.46, 'In stock'),
 (21, 'IN HER WAKE', 12.84, 'In stock'),
 (31,
  'THE FOUR AGREEMENTS: A PRACTICAL GUIDE TO PERSONAL FREEDOM',
  17.66,
  'In stock'),
 (35, "SOPHIE'S WORLD", 15.94, 'In stock'),
 (37, 'MAUDE (1883-1993):SHE GREW UP WITH THE COUNTRY', 18.02, 'In stock'),
 (38, 'IN A DARK, DARK WOOD', 19.63, 'In stock'),
 (48, 'UNTITLED COLLECTION: SABBATH POEMS 2014', 14.27, 'In stock'),
 (50, 'UNICORN TRACKS', 18.78, 'In stock'),
 (52,
  'TSUBASA: WORLD CHRONICLE 2 (TSUBASA WORLD CHRONICLE #2)',
  16.28,
  'In stock'),
 (54, 'THIS ONE SUMMER', 19.49, 'In stock'),
 (55, 'THIRST', 17.27, 'In stock'),
 (56, 'THE TORCH IS PASSED: A HARDING FAMILY STORY', 19.09, 'In stock'),
 (65,
  'THE LIFE-CHANGING MAGIC OF TIDYING UP: THE JAPANESE ART OF DECLUTTERING AND ORGANIZING',


#### `ORDER BY`
##### Consulta com ordenação: Top 5 livros mais caros

In [43]:
cursor.execute("SELECT * FROM books ORDER BY price DESC")
cursor.fetchmany(5)

[(69, 'THE DEATH OF HUMANITY: AND THE CASE FOR LIFE', 58.11, 'In stock'),
 (41, 'SLOW STATES OF COLLAPSE: POEMS', 57.31, 'In stock'),
 (16,
  'OUR BAND COULD BE YOUR LIFE: SCENES FROM THE AMERICAN INDIE UNDERGROUND, 1981-1991',
  57.25,
  'In stock'),
 (59, 'THE PAST NEVER ENDS', 56.5, 'In stock'),
 (58,
  'THE PIONEER WOMAN COOKS: DINNERTIME: COMFORT CLASSICS, FREEZER FOOD, 16-MINUTE MEALS, AND OTHER DELICIOUS WAYS TO SOLVE SUPPER!',
  56.41,
  'In stock')]

#### `GROUP BY`
##### Consulta com agregação: Quantidade de livros por situação de estoque

In [44]:
query = """
  SELECT title, stock, COUNT(stock)
  FROM books
  GROUP BY title, stock
"""

cursor.execute(query)
cursor.fetchall()

[('#HIGHERSELFIE: WAKE UP YOUR LIFE. FREE YOUR SOUL. FIND YOUR TRIBE.',
  'In stock',
  1),
 ('A LIGHT IN THE ATTIC', 'In stock', 1),
 ('ALADDIN AND HIS WONDERFUL LAMP', 'In stock', 1),
 ("AMERICA'S CRADLE OF QUARTERBACKS: WESTERN PENNSYLVANIA'S FOOTBALL FACTORY FROM JOHNNY UNITAS TO JOE MONTANA",
  'In stock',
  1),
 ('BEHIND CLOSED DOORS', 'In stock', 1),
 ('BIRDSONG: A STORY IN PICTURES', 'In stock', 1),
 ('BLACK DUST', 'In stock', 1),
 ('CHASE ME (PARIS NIGHTS #2)', 'In stock', 1),
 ('FOOLPROOF PRESERVING: A GUIDE TO SMALL BATCH JAMS, JELLIES, PICKLES, CONDIMENTS, AND MORE: A FOOLPROOF GUIDE TO MAKING SMALL BATCH JAMS, JELLIES, PICKLES, CONDIMENTS, AND MORE',
  'In stock',
  1),
 ('HOW MUSIC WORKS', 'In stock', 1),
 ('IN A DARK, DARK WOOD', 'In stock', 1),
 ('IN HER WAKE', 'In stock', 1),
 ("IT'S ONLY THE HIMALAYAS", 'In stock', 1),
 ('LIBERTARIANISM FOR BEGINNERS', 'In stock', 1),
 ('MAUDE (1883-1993):SHE GREW UP WITH THE COUNTRY', 'In stock', 1),
 ('MESAERION: THE BEST SCIENCE FI

#### `DISTINCT`
##### Consulta com filtro: Contagem de dados distintos

In [45]:
cursor.execute("SELECT COUNT(DISTINCT title) FROM books")
cursor.fetchall()

[(80,)]

#### `HAVING COUNT(*)`
##### Consulta com filtro: Contagem de dados repetidos

In [46]:
query = """
  SELECT
    price
  FROM books
  GROUP BY price
  HAVING COUNT(*) > 1
"""
cursor.execute(query)
cursor.fetchall()

[(44.18,)]

#### `AVG`
##### Consulta com agregação: Valor médio dos livros

In [47]:
cursor.execute("SELECT avg(price) FROM books")
cursor.fetchone()

(35.708375000000004,)

##### Consulta com agregação: Valor médio dos livros  (com arredondamento)

In [48]:
cursor.execute("SELECT ROUND(AVG(price),3) FROM books")
cursor.fetchone()

(35.708,)

#### `MAX`
##### Consulta com agregação: Maior valor entre os livros `MAX`

In [49]:
cursor.execute("SELECT MAX(price) FROM books")
cursor.fetchone()

(58.11,)

#### `MIN`
##### Consulta com agregação: Menor valor entre os livros

In [50]:
cursor.execute("SELECT MIN(price) FROM books")
cursor.fetchone()

(12.84,)

#### `SUM`
##### Consulta com agregação: Soma do valor dos livros

In [51]:
cursor.execute("SELECT SUM(price) FROM books")
cursor.fetchone()

(2856.67,)

##### Consulta com agregação: Soma do valor dos livros em estoque

In [52]:
cursor.execute("SELECT SUM(price) FROM books WHERE stock = 'In stock'")
cursor.fetchone()

(2856.67,)

##### Consulta com agregação: Soma do valor dos livros sem estoque

In [53]:
cursor.execute("SELECT SUM(price) FROM books WHERE stock <> 'In stock'")
cursor.fetchone()

(None,)

##### Consulta de multiplos valores: Valor Máximo, Medio e Mínimo

In [54]:
cursor.execute("SELECT MAX(price), AVG(price), MIN(price) FROM books")
max, min, mean = cursor.fetchone()
max, min, mean

(58.11, 35.708375000000004, 12.84)

##### Consulta com Ranking por Preço

In [55]:
query = """
  SELECT  title,
          price,
          RANK() OVER(ORDER BY price DESC) AS rank_price
  FROM books
"""
cursor.execute(query)
cursor.fetchall()

[('THE DEATH OF HUMANITY: AND THE CASE FOR LIFE', 58.11, 1),
 ('SLOW STATES OF COLLAPSE: POEMS', 57.31, 2),
 ('OUR BAND COULD BE YOUR LIFE: SCENES FROM THE AMERICAN INDIE UNDERGROUND, 1981-1991',
  57.25,
  3),
 ('THE PAST NEVER ENDS', 56.5, 4),
 ('THE PIONEER WOMAN COOKS: DINNERTIME: COMFORT CLASSICS, FREEZER FOOD, 16-MINUTE MEALS, AND OTHER DELICIOUS WAYS TO SOLVE SUPPER!',
  56.41,
  5),
 ('THE SECRET OF DREADWILLOW CARSE', 56.13, 6),
 ('THE ELECTRIC PENCIL: DRAWINGS FROM INSIDE STATE HOSPITAL NO. 3', 56.06, 7),
 ('BIRDSONG: A STORY IN PICTURES', 54.64, 8),
 ('SAPIENS: A BRIEF HISTORY OF HUMANKIND', 54.23, 9),
 ('THE MURDER THAT NEVER WAS (FORENSIC INSTINCTS #5)', 54.11, 10),
 ('TIPPING THE VELVET', 53.74, 11),
 ('ALADDIN AND HIS WONDERFUL LAMP', 53.13, 12),
 ("SCOTT PILGRIM'S PRECIOUS LITTLE LIFE (SCOTT PILGRIM #1)", 52.29, 13),
 ('BEHIND CLOSED DOORS', 52.22, 14),
 ('THE BLACK MARIA', 52.15, 15),
 ('A LIGHT IN THE ATTIC', 51.77, 16),
 ('LIBERTARIANISM FOR BEGINNERS', 51.33, 17),
 

#### 12: JOINS – Relacionando tabelas

JOINS permitem **combinar dados de várias tabelas**.  
Vamos criar outras tabelas para exemplificar.

##### 12.1 Criar tabela de vendas

In [56]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS sells (
  id INTEGER PRIMARY KEY AUTOINCREMENT,
  book_id INTEGER,
  sell_date DATE,
  amount INTEGER,
  total_value REAL,
  client_name TEXT,
  FOREIGN KEY (book_id) REFERENCES books(id)
);
''')
conn.commit()
print("Success")


Success


##### Verifica o esquema da tabela (estrutura das colunas)

##### 12.2: Inserir dados de exemplo

In [57]:
sells_sample = [
(1, '2025-01-15', 2, 'João Silva'),
(5, '2025-01-16', 1, 'Maria Santos'),
(10, '2025-01-17', 3, 'Pedro Costa'),
(15, '2025-01-18', 1, 'Ana Oliveira'),
(20, '2025-01-19', 2, 'Carlos Lima'),
(25, '2025-01-20', 1, 'Juliana Pereira'),
(30, '2025-01-21', 2, 'Roberto Alves'),
(35, '2025-01-22', 1, 'Fernanda Dias'),
(40, '2025-01-23', 3, 'Ricardo Martins'),
(45, '2025-01-24', 1, 'Patrícia Souza')
]

for book_id, sell_date, amount, client_name in sells_sample:
  # Buscar preço
  cursor.execute("SELECT price FROM books WHERE id = ?", (book_id,))
  result = cursor.fetchone()
  total_value = round(result[0] * amount,2)

  cursor.execute('''
    INSERT INTO sells (book_id, sell_date, amount, total_value, client_name)
    VALUES (?, ?, ?, ?, ?)
  ''', (book_id, sell_date, amount, total_value, client_name))

conn.commit()
print(f"{len(sells_sample)} vendas inseridas com sucesso!")


10 vendas inseridas com sucesso!


##### Verifica dados inseridos

In [58]:
cursor.execute("SELECT * FROM sells")
cursor.fetchall()

[(1, 1, '2025-01-15', 2, 103.54, 'João Silva'),
 (2, 5, '2025-01-16', 1, 54.23, 'Maria Santos'),
 (3, 10, '2025-01-17', 3, 156.45, 'Pedro Costa'),
 (4, 15, '2025-01-18', 1, 35.02, 'Ana Oliveira'),
 (5, 20, '2025-01-19', 2, 90.34, 'Carlos Lima'),
 (6, 25, '2025-01-20', 1, 34.53, 'Juliana Pereira'),
 (7, 30, '2025-01-21', 2, 88.36, 'Roberto Alves'),
 (8, 35, '2025-01-22', 1, 15.94, 'Fernanda Dias'),
 (9, 40, '2025-01-23', 3, 100.89, 'Ricardo Martins'),
 (10, 45, '2025-01-24', 1, 45.07, 'Patrícia Souza')]

##### 12.3: Exemplos de JOINs com a tabela existente

In [63]:
query = """
  SELECT  book.title,
          sell.client_name,
          sell.sell_date
  FROM books book
  JOIN sells sell
    ON book.id = sell.book_id
"""
cursor.execute(query)
#cursor.fetchall()


for row in  cursor.fetchall():
  print(row[0])


A LIGHT IN THE ATTIC
SAPIENS: A BRIEF HISTORY OF HUMANKIND
THE BLACK MARIA
RIP IT UP AND START AGAIN
IT'S ONLY THE HIMALAYAS
BLACK DUST
WALL AND PIECE
SOPHIE'S WORLD
YOU CAN'T BURY THEM ALL: POEMS
WITHOUT BORDERS (WANDERLOVE #1)


In [67]:
query = """
  SELECT  book.title,
          sell.client_name,
          sell.sell_date,
          COALESCE(sell.amount, 0) AS amount
  FROM books book
  LEFT JOIN sells sell
    ON book.id = sell.book_id
"""
cursor.execute(query)
#cursor.fetchall()


for row in  cursor.fetchall():
  print(row[3], row[0])


2 A LIGHT IN THE ATTIC
0 TIPPING THE VELVET
0 SOUMISSION
0 SHARP OBJECTS
1 SAPIENS: A BRIEF HISTORY OF HUMANKIND
0 THE REQUIEM RED
0 THE DIRTY LITTLE SECRETS OF GETTING YOUR DREAM JOB
0 THE COMING WOMAN: A NOVEL BASED ON THE LIFE OF THE INFAMOUS FEMINIST, VICTORIA WOODHULL
0 THE BOYS IN THE BOAT: NINE AMERICANS AND THEIR EPIC QUEST FOR GOLD AT THE 1936 BERLIN OLYMPICS
3 THE BLACK MARIA
0 STARVING HEARTS (TRIANGULAR TRADE TRILOGY, #1)
0 SHAKESPEARE'S SONNETS
0 SET ME FREE
0 SCOTT PILGRIM'S PRECIOUS LITTLE LIFE (SCOTT PILGRIM #1)
1 RIP IT UP AND START AGAIN
0 OUR BAND COULD BE YOUR LIFE: SCENES FROM THE AMERICAN INDIE UNDERGROUND, 1981-1991
0 OLIO
0 MESAERION: THE BEST SCIENCE FICTION STORIES 1800-1849
0 LIBERTARIANISM FOR BEGINNERS
2 IT'S ONLY THE HIMALAYAS
0 IN HER WAKE
0 HOW MUSIC WORKS
0 FOOLPROOF PRESERVING: A GUIDE TO SMALL BATCH JAMS, JELLIES, PICKLES, CONDIMENTS, AND MORE: A FOOLPROOF GUIDE TO MAKING SMALL BATCH JAMS, JELLIES, PICKLES, CONDIMENTS, AND MORE
0 CHASE ME (PARIS NIGHT

In [69]:
# 3. JOIN com agregação - Total de vendas por livro

cursor.execute('''
SELECT
    books.title,
    books.price,
    COUNT(sells.id) as total_vendas,
    SUM(sells.amount) as total_unidades,
    SUM(sells.total_value) as valor_total
FROM books
LEFT JOIN sells ON books.rowid = sells.book_id
GROUP BY books.rowid
ORDER BY total_value DESC NULLS LAST;
''')

for row in cursor.fetchall():
    print(f"Título: {row[0]:<40}")
    print(f"Preço: R$ {row[1]:6.2f}")
    print(f" Vendas: {row[2]} transações")
    print(f"Unidades: {row[3] or 0}")
    print(f"Total: R$ {row[4] or 0:.2f}")
    print()

Título: THE BLACK MARIA                         
Preço: R$  52.15
 Vendas: 1 transações
Unidades: 3
Total: R$ 156.45

Título: A LIGHT IN THE ATTIC                    
Preço: R$  51.77
 Vendas: 1 transações
Unidades: 2
Total: R$ 103.54

Título: YOU CAN'T BURY THEM ALL: POEMS          
Preço: R$  33.63
 Vendas: 1 transações
Unidades: 3
Total: R$ 100.89

Título: IT'S ONLY THE HIMALAYAS                 
Preço: R$  45.17
 Vendas: 1 transações
Unidades: 2
Total: R$ 90.34

Título: WALL AND PIECE                          
Preço: R$  44.18
 Vendas: 1 transações
Unidades: 2
Total: R$ 88.36

Título: SAPIENS: A BRIEF HISTORY OF HUMANKIND   
Preço: R$  54.23
 Vendas: 1 transações
Unidades: 1
Total: R$ 54.23

Título: WITHOUT BORDERS (WANDERLOVE #1)         
Preço: R$  45.07
 Vendas: 1 transações
Unidades: 1
Total: R$ 45.07

Título: RIP IT UP AND START AGAIN               
Preço: R$  35.02
 Vendas: 1 transações
Unidades: 1
Total: R$ 35.02

Título: BLACK DUST                              
Preço: R$  3

In [76]:
# 4. JOIN com filtro WHERE
print("JOIN com Filtro - Livros em Estoque com Vendas:")

cursor.execute('''
SELECT
    books.title,
    books.stock,
    sells.amount,
    sells.sell_date,
    sells.client_name
FROM books
INNER JOIN sells ON books.rowid = sells.book_id
WHERE books.stock = 'In stock'
ORDER BY sells.sell_date DESC;
''')

for row in cursor.fetchall():
    print(f"Título: {row[0]}")
    print(f"Estoque: {row[1]}")
    print(f"Quantidade: {row[2]}")
    print(f"Data: {row[3]}")
    print(f"Cliente: {row[4]}")
    print()

JOIN com Filtro - Livros em Estoque com Vendas:
Título: WITHOUT BORDERS (WANDERLOVE #1)
Estoque: In stock
Quantidade: 1
Data: 2025-01-24
Cliente: Patrícia Souza

Título: YOU CAN'T BURY THEM ALL: POEMS
Estoque: In stock
Quantidade: 3
Data: 2025-01-23
Cliente: Ricardo Martins

Título: SOPHIE'S WORLD
Estoque: In stock
Quantidade: 1
Data: 2025-01-22
Cliente: Fernanda Dias

Título: WALL AND PIECE
Estoque: In stock
Quantidade: 2
Data: 2025-01-21
Cliente: Roberto Alves

Título: BLACK DUST
Estoque: In stock
Quantidade: 1
Data: 2025-01-20
Cliente: Juliana Pereira

Título: IT'S ONLY THE HIMALAYAS
Estoque: In stock
Quantidade: 2
Data: 2025-01-19
Cliente: Carlos Lima

Título: RIP IT UP AND START AGAIN
Estoque: In stock
Quantidade: 1
Data: 2025-01-18
Cliente: Ana Oliveira

Título: THE BLACK MARIA
Estoque: In stock
Quantidade: 3
Data: 2025-01-17
Cliente: Pedro Costa

Título: SAPIENS: A BRIEF HISTORY OF HUMANKIND
Estoque: In stock
Quantidade: 1
Data: 2025-01-16
Cliente: Maria Santos

Título: A LIGHT 

#### 13: Transações – BEGIN, COMMIT e ROLLBACK

Uma transação é um conjunto de operações que devem ser executadas juntas.  

- `BEGIN` → inicia a transação  
- `COMMIT` → confirma as alterações  
- `ROLLBACK` → desfaz as alterações (caso de erro)  

In [82]:
# Exemplo de transação com ROLLBACK
try:
    conn.execute("BEGIN;") # Inicia a trasação
    cursor.execute("INSERT INTO books (title, price, stock) VALUES ('Super Python', 99.90, 'Out of stock');") # Cria novo registro
    #raise Exception("Erro simulado!")  # Força erro
    conn.commit() # Se chegasse aqui, faria commit
except:
    conn.rollback() # Reverte em caso de erro
    print("Transação revertida por erro")

#### Passo 13: Encerra conexão com banco de dados

In [83]:
conn.close()